# A global model in five steps

A global model balances, averaged over a chamber, the density of every species and the
electron energy. This tutorial builds the simplest one, an argon discharge with one ion
species, checks it against the closed form of the textbook model (Lieberman and
Lichtenberg, *Principles of Plasma Discharges*, ch. 10), and looks at every term of its
balances. The cross sections are illustrative step functions, not argon data.

## 1. Species and reactions

Rates come from cross sections averaged over a Maxwellian, with plasmacoll.

In [ ]:
from plasmacoll import CrossSection
from plasmacoll.constants import ELECTRON_MASS

from wellbalanced import ELECTRON, Chamber, GlobalModel, Reaction, Species

argon = Species.atomic("Ar", 39.948)
argon_ion = Species.atomic("Ar+", 39.948, 1)
ionization = Reaction.from_cross_section(
    ("e", "Ar"),
    ("e", "e", "Ar+"),
    CrossSection.constant(2.0e-20, threshold=15.76),
    ELECTRON_MASS,
    electron_energy_loss=15.76,
    name="ionization",
)
[ionization.coefficient(te, 0.025) for te in (1.0, 3.0, 10.0)]  # m^3/s

## 2. The chamber and the model

One litre, 600 cm² of wall, ions leaving at the Bohm speed with an edge-to-centre ratio of 0.5; 100 W absorbed by the electrons; the gas held at 10²⁰ m⁻³.

In [ ]:
chamber = Chamber(volume=1.0e-3, wall_area=0.06, h_factor=0.5)
model = GlobalModel(
    species=[ELECTRON, argon, argon_ion],
    reactions=[ionization],
    chamber=chamber,
    power=100.0,
    fixed={"Ar": 1.0e20},
)
state = model.steady_state()
state.converged, state.electron_temperature, state.electron_density

## 3. Every term of the balances

At steady state the electron power terms (W) sum to zero, and every species' rate of change vanishes.

In [ ]:
state.power

In [ ]:
sum(state.power.values()), state.particle_balance

## 4. The closed form

The particle balance alone sets the electron temperature, $K_{iz}(T_e)\, n_g = h u_B A / V$; the power balance then sets the density.

In [ ]:
import math

from plasmacoll.constants import ELEMENTARY_CHARGE as e
from scipy.optimize import brentq


def bohm(te):
    return math.sqrt(e * te / argon_ion.mass)


te = brentq(
    lambda t: ionization.coefficient(t, 0.025) * 1.0e20 - bohm(t) * 0.5 * 0.06 / 1.0e-3,
    0.5,
    50.0,
)
te, state.electron_temperature

## 5. A scan

Each steady state starts from the last. The electron temperature does not depend on the power, the density is proportional to it.

In [ ]:
states = model.scan("power", [50.0, 100.0, 200.0, 400.0])
[(s.electron_temperature, s.electron_density) for s in states]